In [0]:
%pip install databricks-vectorsearch --quiet
%restart_python

In [0]:
CATALOG = "3dt2ndteam5"
SCHEMA = "cwe"

# LLM_MODEL = "databricks-claude-sonnet-4-6"
LLM_MODEL = "databricks-claude-haiku-4-5" # for testing: Sonnet is high cost

In [0]:
from databricks.vector_search.client import VectorSearchClient
vsc = VectorSearchClient()

ENDPOINT = "cwe-weaknesses-vector-search"
IDX_NAME = f"{CATALOG}.{SCHEMA}.cwe_weaknesses_vs_idx"

index = vsc.get_index(endpoint_name=ENDPOINT, index_name=IDX_NAME)

In [0]:
spark.sql(f"SELECT * FROM {CATALOG}.{SCHEMA}.cwe_weaknesses_silver LIMIT 10").display()

In [0]:
cols = spark.sql("""select column_name from system.information_schema.columns where table_name='cwe_weaknesses_silver' and not column_name = 'search_text'""")
cols = [doc['column_name'] for doc in cols.collect()]

In [0]:
import mlflow.deployments 
client = mlflow.deployments.get_deploy_client("databricks")

def rag_answer(query):
    search_results = index.similarity_search(
        query_text=query,
        columns=cols,
        num_results=10,
    )

    context_docs = search_results.get("result").get("data_array")
    
    # Response Query Formatting
    formatted_context = "### Related CWE Information\n\n"
    for i, doc in enumerate(context_docs, 1):
        formatted_context += f"**Document {i}:**\n"
        for col_idx, col_name in enumerate(cols):
            value = doc[col_idx]
            if value and str(value).strip():
                display_name = col_name.replace('_', ' ').title()
                formatted_context += f"- {display_name}: {value}\n"
        formatted_context += "\n"
    
    # LLM에게 질문과 함께 컨텍스트 전달
    response = client.predict(
        endpoint=LLM_MODEL,
        inputs={
            "messages": [
                {
                    "role": "system",
                    "content": """You are a security analyst specializing in code vulnerabilities and CWE (Common Weakness Enumeration).
                    Use the provided CWE information to give accurate and detailed answers about vulnerabilities.""",
                },
                {
                    "role": "user",
                    "content": f"""{formatted_context}

                    Based on the above CWE information, please answer the following question:
                    {query}"""
                }
            ],
            "max_tokens": 1024,
            "temperature": 0.7,
        }
    )
    
    return response.choices[0]['message']['content']

In [0]:
# Code snippet과 취약점 여부에 따른 동적 query 생성 및 분석
code_snippet = """
    $username = GetCurrentUser();
    $state = GetStateData($username);
    if (defined($state)) {
    $uid = ExtractUserID($state);
    }

    # do stuff
    if ($uid == 0) {
    DoAdminThings();
    }
"""

is_vulnerable = True  # 취약한 경우: True, 안전한 경우: False

# Query 동적 생성
if is_vulnerable:
    dynamic_query = f"""
    Analyze the following code snippet for vulnerabilities:
    
    ```
    {code_snippet}
    ```
    
    This code is marked as VULNERABLE. 
    What specific CWE vulnerabilities does this code have? 
    Please explain the vulnerability in detail and how it could be exploited.
    """
else:
    dynamic_query = f"""
    Analyze the following code snippet:
    
    ```
    {code_snippet}
    ```
    
    This code is marked as NOT VULNERABLE.
    Please explain why this code is safe from vulnerabilities, 
    referencing specific security practices implemented in the code.
    """

print(f"Vulnerability Status: {'VULNERABLE' if is_vulnerable else 'NOT VULNERABLE'}")
print("\n" + "="*80 + "\n")
answer = rag_answer(dynamic_query)
print(answer)